In [1]:
import joblib
import numpy as np
import pandas as pd

from dvf.config import settings
from dvf.model.features import preparer, separer_temporellement
from dvf.model.train import DATE_BASCULE

ventes = pd.read_parquet(settings.processed_data_dir / "ventes_33.parquet")
_, test = separer_temporellement(ventes, DATE_BASCULE)
modele = joblib.load(settings.models_dir / "modele.joblib")

variables, reel = preparer(test)
predit = modele.predict(variables)
erreur = np.abs((reel.to_numpy() - predit) / reel.to_numpy())

print(f"MAPE moyen  : {erreur.mean():.1%}")
print(f"MAPE médian : {np.median(erreur):.1%}")
print(f"90e centile : {np.quantile(erreur, 0.90):.1%}")
print(f"99e centile : {np.quantile(erreur, 0.99):.1%}")

diag = test.assign(erreur=erreur)
diag["tranche"] = pd.qcut(diag["prix"], 5, labels=["très bas", "bas", "moyen", "haut", "très haut"])
diag.groupby("tranche", observed=True).agg(
    erreur_mediane=("erreur", "median"),
    prix_median=("prix", "median"),
    n=("erreur", "size"),
)

MAPE moyen  : 35.5%
MAPE médian : 17.0%
90e centile : 64.5%
99e centile : 320.8%


,erreur_mediane,prix_median,n
tranche,,,
très bas,0.338574,105000.0,3346
bas,0.168186,165000.0,3354
moyen,0.137525,230000.0,3341
haut,0.147232,311050.0,3345
très haut,0.149946,505000.0,3343
